# 数据分割：训练集、验证集、测试集的划分之道
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：01_数据预处理 | 源文件：数据分割.py | 核心功能：7种数据分割策略的完整演示

## 概述

数据分割是机器学习工作流中**最容易被低估**的环节。很多人把精力花在调模型、调参数上，却忽略了数据分割的质量——殊不知，一个糟糕的分割可能让再好的模型也发挥不出来。

这个脚本演示了 7 种主流分割方法，从最简单的 	rain_test_split 到专门针对时间序列的 TimeSeriesSplit。每种方法都有其适用场景，选错了可能导致**数据泄露**——让测试集的信息"偷偷"流入训练过程，造成虚假的高分。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split,
    KFold,
# --- 继续 ---
    StratifiedKFold,
    LeaveOneOut,
    ShuffleSplit,
    TimeSeriesSplit,
)

## 数学原理

### 1. 留出法（Hold-out）的无偏性分析

留出法将数据集 $D$ 随机划分为训练集 $S$ 和测试集 $T$，满足 $S \cup T = D$，$S \cap T = \emptyset$。设测试集占比为 $\beta = |T|/|D|$。

**代码对应**：`train_test_split(X, y, test_size=0.2)` 中 `test_size=0.2` 即 $\beta = 0.2$。

泛化误差的估计为：

$$\hat{R}(f) = \frac{1}{|T|} \sum_{(x_i, y_i) \in T} L(y_i, f(x_i))$$

其中 $L$ 为损失函数。该估计的期望等于真实泛化误差 $R(f) = \mathbb{E}_{(x,y) \sim \mathcal{D}}[L(y, f(x))]$，但方差取决于 $|T|$：

$$\text{Var}[\hat{R}(f)] = \frac{\sigma^2}{|T|}$$

当 $|T|$ 过小时，评估结果方差大、不稳定。经验上 $|T|/|D| \in [0.15, 0.3]$ 是常见选择。

### 2. 分层抽样（Stratified Sampling）的数学保证

**代码对应**：`train_test_split(..., stratify=y)` 确保各类别比例一致。

设总体中第 $k$ 类的比例为 $\pi_k = N_k / N$。分层抽样要求每个子集中：

$$\hat{\pi}_k^{(S)} = \frac{|S \cap C_k|}{|S|} \approx \pi_k, \quad \hat{\pi}_k^{(T)} = \frac{|T \cap C_k|}{|T|} \approx \pi_k$$

分层抽样相比简单随机抽样，分类准确率估计的方差更小。设类别 $k$ 上的方差为 $\sigma_k^2$，分层估计的方差为：

$$\text{Var}_{\text{strat}} = \sum_{k=1}^{K} \pi_k^2 \frac{\sigma_k^2}{n_k}$$

而简单随机抽样的方差为：

$$\text{Var}_{\text{simple}} = \frac{1}{N} \left( \sum_{k=1}^{K} \pi_k \sigma_k^2 + \sum_{k=1}^{K} \pi_k (\mu_k - \mu)^2 \right)$$

由于第二项（组间方差）为正，分层抽样方差始终更小或相等。这就是分类任务应始终使用 `stratify=y` 的数学原因。

### 3. K 折交叉验证的偏差-方差权衡

**代码对应**：`KFold(n_splits=5, shuffle=True)` 将数据分为 $K=5$ 折。

将数据均匀分为 $K$ 份 $D_1, D_2, \ldots, D_K$，第 $k$ 折的验证误差为：

$$\hat{R}_k = \frac{1}{|D_k|} \sum_{(x_i, y_i) \in D_k} L(y_i, \hat{f}^{(-k)}(x_i))$$

其中 $\hat{f}^{(-k)}$ 是在 $D \setminus D_k$ 上训练的模型。K 折交叉验证的估计为：

$$\hat{R}_{CV} = \frac{1}{K} \sum_{k=1}^{K} \hat{R}_k$$

**偏差分析**：$\hat{f}^{(-k)}$ 的训练集大小为 $\frac{K-1}{K} N$，比全数据集小。因此 $\hat{R}_{CV}$ 存在**乐观偏差**（optimistic bias），因为训练集偏小导致模型性能偏低。$K$ 越大，训练集越接近全集，偏差越小。

**方差分析**：$K$ 越大，各折的训练集重叠越多，$\hat{R}_k$ 之间的相关性越高，导致方差增大。$K=5$ 或 $K=10$ 是偏差-方差权衡的常用折中。

| $K$ 值 | 偏差 | 方差 | 训练集比例 | 计算成本 |
|--------|------|------|-----------|---------|
| 2 | 高 | 低 | 50% | 低 |
| 5 | 中 | 中 | 80% | 中 |
| 10 | 低 | 较高 | 90% | 高 |
| $N$ (LOO) | 近无偏 | 最高 | $\frac{N-1}{N}$ | 最高 |

### 4. 留一法（Leave-One-Out）的极限情况

**代码对应**：`LeaveOneOut()` 等价于 $K=N$ 的交叉验证。

$$\hat{R}_{LOO} = \frac{1}{N} \sum_{i=1}^{N} L(y_i, \hat{f}^{(-i)}(x_i))$$

LOO 的偏差近似无偏（训练集大小为 $N-1 \approx N$），但方差最大——各折训练集高度重叠（$N-2$ 个样本相同），导致 $\hat{R}_k$ 之间高度正相关。

对于线性回归，LOO 误差有高效计算公式（PRESS 统计量）：

$$\hat{R}_{LOO} = \frac{1}{N} \sum_{i=1}^{N} \left(\frac{e_i}{1 - h_{ii}}\right)^2$$

其中 $e_i = y_i - \hat{y}_i$ 为普通残差，$h_{ii}$ 为帽子矩阵 $\mathbf{H} = \mathbf{X}(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T$ 的第 $i$ 个对角元素（杠杆值）。只需一次模型拟合即可计算所有 LOO 残差。

### 5. 时间序列分割的信息泄露约束

**代码对应**：`TimeSeriesSplit(n_splits=5)` 确保训练集始终在时间上早于验证集。

对于时间序列 $\{x_1, x_2, \ldots, x_N\}$，第 $k$ 折的划分满足：

$$\text{训练集} = \{x_1, \ldots, x_{t_k}\}, \quad \text{验证集} = \{x_{t_k+1}, \ldots, x_{t_{k+1}}\}$$

其中 $t_1 < t_2 < \cdots < t_K$。这个约束确保：

$$\forall i \in \text{训练集}, \forall j \in \text{验证集}: \quad i < j$$

如果违反此约束（使用随机分割），自相关结构会导致未来信息泄露：

$$\text{Cov}(x_t, x_{t+h}) = \gamma(h) \neq 0$$

模型会利用 $\gamma(h)$ 中的未来信息"作弊"，导致交叉验证评分虚高但实际部署失败。

### 6. Bootstrap 采样的 OOB 估计

**代码对应**：`ShuffleSplit` 与 Bootstrap 采样思想类似。

Bootstrap 有放回采样 $N$ 次，单个样本**不被抽中**的概率为：

$$P(\text{未被抽中}) = \left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} e^{-1} \approx 0.368$$

即约 **36.8%** 的样本不在训练集中（Out-of-Bag, OOB）。OOB 误差估计为：

$$\hat{R}_{OOB} = \frac{1}{|\text{OOB}|} \sum_{i \in \text{OOB}} L(y_i, \hat{f}^{(-i)}(x_i))$$

其中 $\hat{f}^{(-i)}$ 是不包含样本 $i$ 的 Bootstrap 训练集上训练的模型。OOB 估计是泛化误差的近似无偏估计。

### 7. 嵌套交叉验证的数学结构

外层 $K_o$ 折评估泛化能力，内层 $K_i$ 折调参：

$$\hat{R}_{\text{nested}} = \frac{1}{K_o} \sum_{j=1}^{K_o} L\left(y^{(j)}, f_{\hat{\lambda}^{(j)}}(x^{(j)})\right)$$

其中 $\hat{\lambda}^{(j)} = \arg\min_{\lambda} \frac{1}{K_i} \sum_{l=1}^{K_i} R_{jl}(\lambda)$ 是内层交叉验证选出的最优超参数。

嵌套交叉验证是超参数调优的**黄金标准**——它对泛化误差的估计几乎没有偏差，因为测试数据从未参与任何建模决策。

## 代码结构

| 方法 | 适用场景 | 核心 API |
|------|----------|----------|
| 简单留出法 | 数据充足时 | 	rain_test_split |
| 三层划分 | 需要独立验证集时 | 两次 	rain_test_split |
| K 折交叉验证 | 数据有限时 | KFold |
| 分层 K 折 | 分类任务（首选） | StratifiedKFold |
| 留一法 | 极小数据集 | LeaveOneOut |
| 随机重复采样 | 需要多次随机评估 | ShuffleSplit |
| 时间序列分割 | 时序数据 | TimeSeriesSplit |

### 1. 生成示例数据

构造示例数据，方便后续可视化算法效果。

In [ ]:
X, y = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_classes=3, n_clusters_per_class=1, random_state=42
)
print(f"数据集形状: X={X.shape}, y={y.shape}")
# --- 输出结果 ---
print(f"类别分布: {np.bincount(y)}")

### 2. 简单留出法（train_test_split）

运行 2. 简单留出法（train_test_split） 的代码，观察算法在该环节的行为。

In [ ]:
# 最基础的分割：按比例随机划分
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y  # stratify 保持类别比例
)
print(f"\n=== 简单留出法 ===")
print(f"训练集: {X_train.shape[0]} 样本, 测试集: {X_test.shape[0]} 样本")
# --- 输出结果 ---
print(f"训练集类别分布: {np.bincount(y_train)}")
print(f"测试集类别分布: {np.bincount(y_test)}")

# 三层划分：训练/验证/测试
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
# --- 赋值/配置 ---
)  # 0.25 × 0.8 = 0.2，所以最终 60/20/20
print(f"\n=== 三层划分 ===")
print(f"训练集: {len(X_train)}, 验证集: {len(X_val)}, 测试集: {len(X_test)}")

### 3. K 折交叉验证（KFold）

运行 3. K 折交叉验证（KFold） 的代码，观察算法在该环节的行为。

In [ ]:
# 将数据均分为 K 份，每次取 1 份作验证，其余作训练
kf = KFold(n_splits=5, shuffle=True, random_state=42)
print(f"\n=== 5 折交叉验证 ===")
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}: 训练={len(train_idx)}, 验证={len(val_idx)}")

### 4. 分层 K 折（StratifiedKFold）

运行 4. 分层 K 折（StratifiedKFold） 的代码，观察算法在该环节的行为。

In [ ]:
# 每折保持类别比例与整体一致，分类任务首选
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print(f"\n=== 分层 5 折交叉验证 ===")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_dist = np.bincount(y[train_idx])
    val_dist = np.bincount(y[val_idx])
# --- 输出结果 ---
    print(f"Fold {fold+1}: 训练={len(train_idx)} {train_dist}, 验证={len(val_idx)} {val_dist}")

### 5. 留一法（LeaveOneOut）

运行 5. 留一法（LeaveOneOut） 的代码，观察算法在该环节的行为。

In [ ]:
# 每次只留 1 个样本作验证，计算量大但偏差小
loo = LeaveOneOut()
print(f"\n=== 留一法 ===")
print(f"总折数: {loo.get_n_splits(X)}（等于样本数 {len(X)}）")
# 仅演示前 3 折
for i, (train_idx, val_idx) in enumerate(loo.split(X)):
    if i >= 3:
        break
    print(f"  Fold {i+1}: 训练={len(train_idx)}, 验证={len(val_idx)} (样本索引={val_idx[0]})")

### 6. 随机重复采样（ShuffleSplit）

运行 6. 随机重复采样（ShuffleSplit） 的代码，观察算法在该环节的行为。

In [ ]:
# 每次随机采样 train_size 样本作训练，剩余作验证，可重复多次
ss = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
print(f"\n=== 随机重复采样 (5 次) ===")
for fold, (train_idx, val_idx) in enumerate(ss.split(X)):
    print(f"Split {fold+1}: 训练={len(train_idx)}, 验证={len(val_idx)}")

### 7. 时间序列分割（TimeSeriesSplit）

运行 7. 时间序列分割（TimeSeriesSplit） 的代码，观察算法在该环节的行为。

In [ ]:
# 保持时间顺序：训练集始终在验证集之前，防止未来信息泄露
tscv = TimeSeriesSplit(n_splits=5)
print(f"\n=== 时间序列分割 ===")
# 模拟时间序列数据
X_ts = np.arange(100).reshape(-1, 1)
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_ts)):
    print(f"Fold {fold+1}: 训练=[0:{train_idx[-1]+1}]({len(train_idx)}), "
          f"验证=[{val_idx[0]}:{val_idx[-1]+1}]({len(val_idx)})")

### 8. 生成数据文件用于后续目录

生成合成数据集，用于演示算法的核心行为。

In [ ]:
# 保存一份干净的训练/测试集供后续算法使用
from sklearn.datasets import load_iris
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris)
print(f"\n=== Iris 数据集分割（供后续使用）===")
# --- 输出结果 ---
print(f"训练集: {X_tr.shape}, 测试集: {X_te.shape}")

print("\n=== 分割策略选择指南 ===")
print("1. 数据充足 → 简单留出法 (train_test_split)")
print("2. 数据有限 → K 折交叉验证 (StratifiedKFold)")
print("3. 分类任务 → 分层 K 折，保持类别平衡")
print("4. 时间序列 → TimeSeriesSplit，严禁打乱顺序")
# --- 输出结果 ---
print("5. 超参调优 → 用验证集/交叉验证；测试集只在最终评估时使用一次")

## 关键代码解释

### 分层抽样 —— 保持类别平衡

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
```

stratify=y 是分类任务的**必备参数**。它确保训练集和测试集的类别比例与原始数据一致。如果不使用分层，当类别不平衡时（比如正样本只占 5%），测试集可能恰好没有正样本，导致评估完全失真。

### 三层划分 —— 最规范的分割方式

```python
X_train_val, X_test = train_test_split(X, y, test_size=0.2, ...)
X_train, X_val = train_test_split(X_train_val, test_size=0.25, ...)
# 最终比例：60% 训练 / 20% 验证 / 20% 测试
```

**为什么要三层？** 训练集用于拟合模型，验证集用于调参和模型选择，测试集**只在最后评估一次**。如果用测试集调参，测试集就变成了"验证集"，你对模型泛化能力的估计就会过于乐观。

### 分层 K 折交叉验证 —— 小数据集的最佳选择

```python
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # 每折的类别比例都与整体一致
```

当数据量有限（比如只有 500 个样本），简单的 80/20 分割会导致测试集只有 100 个样本，评估结果方差很大。K 折交叉验证让每个样本都被验证过一次，得到更稳定的评估。

### 时间序列分割 —— 严禁打乱顺序

```python
tscv = TimeSeriesSplit(n_splits=5)
for fold, (train_idx, val_idx) in enumerate(tscv.split(X_ts)):
    # 训练集始终在验证集之前
```

时间序列数据有时间依赖性。如果用随机分割，模型可能"看到未来"——用明天的数据预测今天，这在实际应用中是不可能的。TimeSeriesSplit 确保训练集始终在时间上早于验证集。

## 使用示例

```python
from sklearn.model_selection import train_test_split, StratifiedKFold

# 简单分割
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 交叉验证
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(X, y):
    model.fit(X[train_idx], y[train_idx])
    score = model.score(X[val_idx], y[val_idx])
```

## 注意事项

1. **随机种子**：始终设置 
andom_state，确保结果可复现
2. **数据泄露**：测试集信息不能以任何形式参与训练过程——包括特征缩放、特征选择、缺失值填充
3. **分层 vs 不分层**：分类任务几乎总是应该使用 stratify
4. **时间序列**：时序数据**绝对不能** shuffle，否则会泄露未来信息
5. **留一法的代价**：n 个样本要训练 n 次，只在数据极少（< 50）时才考虑

## 总结与延伸

以上代码展示了 **数据分割** 的完整流程。

**解读要点**：
- 观察处理前后数据的 **统计分布变化**
- 关注缺失值填充策略对下游模型的影响
- 对比不同缩放方法的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **GroupKFold**：当数据有分组结构（如同一患者的多个记录）时，确保同组数据不同时出现在训练集和验证集
- **RepeatedStratifiedKFold**：重复多次 K 折交叉验证，进一步降低评估方差
- **嵌套交叉验证**：外层评估模型泛化能力，内层调参，是超参数调优的黄金标准
- **分层回归**：对回归任务，可以先将连续目标变量分桶，再做分层抽样
